# SNNs-auf-GPUs -- Colab test run

Builds the environment, then runs `learning/main.py` end to end (train + test) -- interactively, same as running it in a local terminal: it'll pause and ask you to pick a dataset and an inference mode, exactly like the prompts you see locally.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better). Run the cells in order (not `Run all`, since the main.py cell will pause waiting for your input).

Clones from the repo referenced in this project's own docs (`docs/roadmap.md`). If your remote/branch differs, edit the URL/branch below before running.


In [ ]:
!nvidia-smi

## 1. Get the code onto this Colab instance

In [ ]:
REPO_URL = "https://github.com/Zuzu3290/SNNs-auf-GPUs.git"
BRANCH = "46-cache_engine"  # edit if you're validating a different branch

!git clone --branch {BRANCH} {REPO_URL} /content/SNNs-auf-GPUs
%cd /content/SNNs-auf-GPUs


In [ ]:
!ls


## 2. Build -- install dependencies

Colab already ships a CUDA-matched `torch` build -- `requirements.txt` only pins `torch>=2.1.0`, so pip leaves Colab's own install alone and just adds everything else.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f"torch          : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

import psutil
print(f"\nphysical cores : {psutil.cpu_count(logical=False)}")
print(f"RAM available  : {psutil.virtual_memory().available / 1024**3:.2f} GB / {psutil.virtual_memory().total / 1024**3:.2f} GB total")

import multiprocessing as mp
print(f"mp start method: {mp.get_context().get_start_method()}  (fork on Colab -- no spawn-reload worker penalty, unlike Windows)")

## 3. Run main.py -- train + test

Run with `%run`, not `!python` -- `!python` launches a separate subprocess whose stdin is never connected to anything, so `input()` there hits EOFError immediately, and the pipeline silently falls back to N-MNIST instead of asking. `%run` executes the script inside this same notebook kernel, where `input()` genuinely works (Colab shows an inline text box) -- so you get the exact same interactive prompts you'd see running `python learning/main.py` in a local terminal: pick a dataset, then pick statistics-only vs. live visualization for inference.

In [ ]:
%run learning/main.py

## 4. See the performance

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/data/training_results.csv")
display(df[["epoch", "train_loss", "train_accuracy", "spike_rate", "spikes_per_neuron_per_inference", "epoch_duration_s"]])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df["epoch"], df["train_loss"], marker="o")
axes[0].set(title="Train loss", xlabel="epoch", ylabel="loss")
axes[1].plot(df["epoch"], df["train_accuracy"] * 100, marker="o", color="tab:green")
axes[1].set(title="Train accuracy", xlabel="epoch", ylabel="%")
plt.tight_layout()
plt.show()